<a href="https://colab.research.google.com/github/2403A52058/NLP_LABASSIGNMENTS/blob/main/NLP_lab_20_2403a52058.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
STEP 1: Install required libraries

This cell installs:
- chromadb → vector database
- sentence-transformers → embedding model
- langchain → optional framework for RAG pipelines
"""

# Install required packages
!pip install chromadb
!pip install sentence-transformers
!pip install langchain

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.4/22.4 MB 55.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 69.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 57.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.5 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.38.0
    Uninstalling opentelem

In [2]:
"""

We import:
- chromadb → to create vector database
- SentenceTransformer → to convert text into embeddings (vectors)
"""

import chromadb
from sentence_transformers import SentenceTransformer

In [3]:
"""


- Client → main interface to interact with ChromaDB
- Collection → similar to a table in traditional databases
  where embeddings (vectors) are stored
"""

# Create ChromaDB client
client = chromadb.Client()

# Create a collection (like a table)
collection = client.create_collection(name="rag_collection")

In [4]:
"""


- SentenceTransformer converts text into numerical vectors
- 'all-MiniLM-L6-v2' is a lightweight and fast embedding model
"""

# Load pre-trained embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [5]:
"""


- These are small text entries used for testing
- Each document has a unique ID
"""

# Sample dataset
documents = [
    "Artificial Intelligence is the simulation of human intelligence.",
    "Machine Learning is a subset of AI.",
    "Deep Learning uses neural networks.",
    "Natural Language Processing deals with text data.",
    "RAG combines retrieval and generation."
]

# Unique IDs for each document
ids = ["doc1", "doc2", "doc3", "doc4", "doc5"]

In [6]:
"""


- The model encodes each document into a vector
- Vectors capture semantic meaning of text
"""

# Convert documents into embeddings (vectors)
embeddings = model.encode(documents)

# Print shape of embeddings
print("Number of embeddings:", len(embeddings))
print("Embedding dimension:", len(embeddings[0]))

Number of embeddings: 5
Embedding dimension: 384


In [7]:
"""


- documents → original text
- embeddings → vector representation
- ids → unique identifiers
"""

# Add data to collection
collection.add(
    documents=documents,
    embeddings=embeddings,
    ids=ids
)

print("Documents stored successfully!")

Documents stored successfully!


In [8]:
"""
- Convert query into embedding
- Retrieve most similar documents using vector similarity
"""

# User query
query = "What is AI?"

# Convert query into embedding
query_embedding = model.encode([query])

# Perform search
results = collection.query(
    query_embeddings=query_embedding,
    n_results=2  # top 2 similar documents
)

# Print raw results
print(results)

{'ids': [['doc1', 'doc2']], 'embeddings': None, 'documents': [['Artificial Intelligence is the simulation of human intelligence.', 'Machine Learning is a subset of AI.']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[None, None]], 'distances': [[0.5604256391525269, 0.700352132320404]]}


In [9]:
"""


- Extract documents from results
- Show most relevant answers
"""

# Display retrieved documents
print("Top matching documents:\n")

for doc in results['documents'][0]:
    print("-", doc)

Top matching documents:

- Artificial Intelligence is the simulation of human intelligence.
- Machine Learning is a subset of AI.


In [10]:
"""


- Takes user query
- Returns top relevant documents
"""

def retrieve(query):
    """
    Retrieve top 2 most relevant documents for a query

    Args:
        query (str): User input question

    Returns:
        list: Relevant documents
    """

    # Convert query to embedding
    query_embedding = model.encode([query])

    # Search in database
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=2
    )

    return results['documents'][0]

# Test function
print(retrieve("Explain machine learning"))

['Machine Learning is a subset of AI.', 'Deep Learning uses neural networks.']


In [11]:
"""


RAG (Retrieval-Augmented Generation) works in 4 steps:

1. User Query
   - User asks a question

2. Convert Query to Embedding
   - Query is transformed into a vector

3. Retrieve Similar Documents
   - Vector database finds closest matching documents

4. Pass to LLM
   - Retrieved documents are sent to a language model
   - Model generates final answer (next lab)
"""

print("RAG workflow explained successfully!")

RAG workflow explained successfully!
